<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
RAG(Retrieval-Augmented Generation) 개념 실습
</div>

로컬 GPU에 올린 **Qwen2.5-0.5B-Instruct(4bit)** 모델을 LangChain의 커스텀 ChatModel로 감싸고,
로컬 문서(HTML, PDF) 로딩 → 분할(Splitter) → 임베딩 → 벡터 스토어 → LCEL 체인까지
RAG의 전체 흐름을 단계별로 실습합니다.

| 단계 | 사용 컴포넌트 |
|---|---|
| 문서 로딩 | `pypdf` + `BeautifulSoup` → `Document` 직접 생성 |
| 문서 분할 | `CharacterTextSplitter` 외 다양한 Splitter |
| 임베딩 | `HuggingFaceEmbeddings` (multilingual-e5-base) |
| 벡터 스토어 | `Chroma` |
| 답변 생성 | `QwenChatModel` (커스텀 ChatModel) |

실습 문서는 실제 보안 권고문 2건입니다.
- `ICSA.html` : CISA의 Hitachi Energy Relion 제품 취약점 권고 (ICSA-25-155-02)
- `mitsubishi.pdf` : Mitsubishi Electric FA 소프트웨어 취약점 권고

# 기본환경 설정

In [ ]:
# %%capture
# %pip install -q -U bitsandbytes accelerate \
#     langchain-core langchain-text-splitters \
#     langchain-huggingface langchain-chroma \
#     sentence-transformers nltk tiktoken lxml \
#     pypdf beautifulsoup4

In [ ]:
%%capture
%pip install -q -U unsloth langchain-core

# GPU와 실행 환경 확인

런타임 유형이 **GPU(T4)** 로 설정되어 있는지 확인합니다.

In [ ]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 GPU: {torch.cuda.get_device_name(0)}")

# 4비트 Qwen2.5 모델 로딩

미리 4bit로 양자화된 모델이라 Colab 무료 T4에서도 가볍게 동작합니다.
공개 모델이므로 Hugging Face 로그인은 필요하지 않습니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

model.eval()

# 커스텀 ChatModel (QwenChatModel)

Hugging Face 모델을 LangChain의 `BaseChatModel`로 감싸면
`invoke()`, `batch()`, `stream()` 등 LangChain 표준 인터페이스와 LCEL 체인에서 그대로 사용할 수 있습니다.

1. `model`, `tokenizer`를 Pydantic 필드로 선언
2. LangChain 메시지를 Qwen 채팅 형식으로 변환
3. `_generate()`에서 일반 응답 생성
4. `_stream()`에서 스트리밍 응답 생성

In [ ]:
import torch
from threading import Thread
from typing import Any, Iterator, List, Optional
from pydantic import ConfigDict

In [ ]:
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult

from transformers import TextIteratorStreamer

In [ ]:
class QwenChatModel(BaseChatModel):
    """Qwen2.5-Instruct를 LangChain ChatModel로 감싸는 클래스."""

    # BaseChatModel은 Pydantic 기반이므로 필드 선언이 필요합니다.
    model: Any
    tokenizer: Any

    max_tokens: int = 512
    do_sample: bool = True
    temperature: float = 0.7
    top_p: float = 0.9

    model_config = ConfigDict(arbitrary_types_allowed=True)

    @property
    def _llm_type(self) -> str:
        return "qwen2.5-custom-chatmodel"

    def _tokenize(self, messages: List[BaseMessage]):
        """LangChain 메시지를 Qwen 채팅 형식으로 변환하고 토큰화합니다."""
        chat = []

        for message in messages:
            if isinstance(message, SystemMessage):
                role = "system"
            elif isinstance(message, HumanMessage):
                role = "user"
            elif isinstance(message, AIMessage):
                role = "assistant"
            else:
                role = "user"

            chat.append({"role": role, "content": message.content})

        inputs = self.tokenizer.apply_chat_template(
            chat,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        return inputs.to(self.model.device)

    def _generation_options(self):
        """model.generate()에 공통으로 전달할 옵션입니다."""
        # RAG에서는 검색된 문맥이 프롬프트에 포함되어 입력이 길어지므로,
        # 전체 길이를 제한하는 max_length 대신 "새로 생성할 토큰 수"를
        # 제한하는 max_new_tokens를 사용해야 안전합니다.
        options = {
            "max_length": self.max_tokens,
            "do_sample": self.do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
        }

        if self.do_sample:
            options["temperature"] = self.temperature
            options["top_p"] = self.top_p

        return options

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        """invoke()와 batch()가 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)
        input_length = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **self._generation_options(),
            )

        new_tokens = outputs[0][input_length:]
        text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        return ChatResult(
            generations=[ChatGeneration(message=AIMessage(content=text))]
        )

    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> Iterator[ChatGenerationChunk]:
        """stream()이 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        thread = Thread(
            target=self.model.generate,
            kwargs={
                **inputs,
                **self._generation_options(),
                "streamer": streamer,
            },
        )
        thread.start()

        for text in streamer:
            chunk = ChatGenerationChunk(
                message=AIMessageChunk(content=text)
            )

            if run_manager:
                run_manager.on_llm_new_token(text, chunk=chunk)

            yield chunk

        thread.join()

In [ ]:
chat_model = QwenChatModel(model=model, tokenizer=tokenizer, max_tokens=2048)

In [ ]:
# 동작 확인
response = chat_model.invoke("RAG가 무엇인지 한 문장으로 설명해 주세요.")
print(response.content)

# Document Load (로컬 문서)

> 💡 **langchain-community를 쓰지 않는 이유**
> 각종 Document Loader가 들어있던 `langchain-community` 패키지는 2026년 5월부로
> 지원이 중단(sunset)되었습니다. LangChain의 현재 방향은 "필요한 통합은 표준 인터페이스에
> 맞춰 직접 구현하거나, 개별 패키지를 사용"하는 것입니다.
>
> 사실 Document Loader의 역할은 단순합니다 — 어떤 소스에서든 텍스트를 읽어
> **`Document(page_content, metadata)`** 객체를 만들어 주는 것뿐입니다.
> 이번 실습에서는 `pypdf`와 `BeautifulSoup`으로 이 과정을 직접 구현하며
> Document의 구조를 정확히 이해해 봅니다.

In [ ]:
# 실습 문서를 res 폴더에 준비합니다.
# Colab이라면 두 파일(ICSA.html, mitsubishi.pdf)이 없을 때 업로드 창이 열립니다.
import os

os.makedirs("res", exist_ok=True)

needed = ["ICSA.html", "mitsubishi.pdf"]
missing = [f for f in needed if not os.path.exists(f"res/{f}")]

if missing:
    print(f"업로드가 필요한 파일: {missing}")
    from google.colab import files
    uploaded = files.upload()  # 파일 선택 창에서 ICSA.html, mitsubishi.pdf 선택
    for name in uploaded:
        os.replace(name, f"res/{name}")

print(os.listdir("res"))

## PDF 로딩 (pypdf)

PDF는 페이지 단위로 텍스트를 추출하고, 페이지마다 하나의 `Document`를 만듭니다.
`metadata`에 출처와 페이지 번호를 남겨 두면, 나중에 검색된 답변의 근거를 추적할 수 있습니다.

In [ ]:
from pypdf import PdfReader
from langchain_core.documents import Document

In [ ]:
reader = PdfReader("res/mitsubishi.pdf")

pdf_docs = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    pdf_docs.append(
        Document(
            page_content=text,
            metadata={"source": "res/mitsubishi.pdf", "page": i + 1},
        )
    )

print(f"PDF 페이지 수: {len(pdf_docs)}")
print(pdf_docs[0].page_content[:300])

## HTML 로딩 (BeautifulSoup)

HTML은 태그를 제거한 본문 텍스트만 추출해서 하나의 `Document`로 만듭니다.
(원본 HTML 문자열은 뒤의 `HTMLHeaderTextSplitter` 실습에서 다시 사용합니다.)

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
with open("res/ICSA.html", "r", encoding="utf-8") as f:
    html_text = f.read()

soup = BeautifulSoup(html_text, "lxml")
plain_text = soup.get_text(separator="\n", strip=True)

html_docs = [
    Document(
        page_content=plain_text,
        metadata={"source": "res/ICSA.html"},
    )
]

print(f"HTML 텍스트 길이: {len(plain_text)}")
print(plain_text[:300])

In [ ]:
# 두 문서를 합쳐 RAG의 입력으로 사용합니다.
raw_docs = pdf_docs + html_docs
print(f"전체 Document 수: {len(raw_docs)}")

# Document Transformer

## CharacterTextSplitter
- 사용자가 separator 인자로 지정한 단일 문자(예: "\n")을 기준으로 분할합니다.
- 사용자가 분할 기준으로 직접 정할 수도 있습니다.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000, # 목표치
    chunk_overlap=0,
    separator="\n", #사용자가 직접 지정 가능
)

In [ ]:
chunk_docs = text_splitter.split_documents(raw_docs)
print(len(chunk_docs))

In [ ]:
print(f"청크 개수: {len(chunk_docs)}")

for i, doc in enumerate(chunk_docs[:2]):
    print(f"\n--- 조각 {i+1} ({len(doc.page_content)}) ---\n{doc.page_content}")

## RecursiveCharacterTextSplitter
- ["\n\n", "\n", " ", ""] 순서로, 큰 단위(문단)부터 작은 단위(단어)로 재귀적으로 분할합니다.
- "" : 문자(character)단위로 분할

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],  # 여러 구분자를 순차적으로 적용
    chunk_size=1000,
    chunk_overlap=10,
)

In [ ]:
chunk_docs = text_splitter.split_documents(raw_docs)

In [ ]:
print(f"청크 개수: {len(chunk_docs)}")

for i, doc in enumerate(chunk_docs[:2]):
    print(f"\n--- 조각 {i+1} ({len(doc.page_content)}) ---\n{doc.page_content}")

## TokenTextSplitter
- 언어 모델의 토큰 단위로 chunk_size 길이에 맞추어 분할합니다.

In [ ]:
from langchain_text_splitters import TokenTextSplitter

In [ ]:
# openai 토크나이저(tiktoken)로 자르기
text_splitter = TokenTextSplitter(
    model_name="gpt-3.5-turbo",
    chunk_size=1000,
    chunk_overlap=0
)

In [ ]:
chunk_docs = text_splitter.split_documents(raw_docs)

In [ ]:
print(f"청크 개수: {len(chunk_docs)}")

for i, doc in enumerate(chunk_docs[:2]):
    print(f"\n--- 조각 {i+1} ({len(doc.page_content)}) ---\n{doc.page_content}")

In [ ]:
# Local Tokenizer로 자르기
# Qwen의 AutoTokenizer는 그대로 전달하면 됩니다.
text_splitter = TokenTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=1000,
    chunk_overlap=0
)

In [ ]:
# Local Tokenizer로 자르기 : 문맥을 좀더 효율적으로 찾아서 분리.
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=1000,
    chunk_overlap=0,             # 필요시 100~200 정도로 조정
    add_start_index=False        # 청크 시작 인덱스가 필요하면 True
)

## NLTKTextSplitter

내부적으로 nltk.sent_tokenize() → Punkt 문장 분리기 사용  
즉, 문장 경계 후보(., ?, !, …) + Punkt 규칙(약어/숫자/대문자 등)을 기반으로 문장을 나눔  

* 기호 후보: ., ?, !, …는 잠정적 문장 끝 후보
* 대문자 시작: 구분 기호 뒤 단어가 대문자로 시작 → 문장 끝일 가능성 ↑
* 약어 인식: Dr., Mr., e.g. 등은 문장 끝이 아님
* 이니셜/다중 점: U.S.A., J. R. R. 같은 패턴은 내부로 처리
* 숫자/소수점: 3.14, No. 5 같은 경우는 문장 끝 아님
* 엘립시스(...): 상황에 따라 문장 끝일 수도 있고 아닐 수도 있음
* 따옴표/괄호: .", !) 같은 닫는 부호는 함께 문장 끝으로 취급
* 특수 패턴: URL, 이메일, 파일명은 경계로 오인하지 않음  

In [ ]:
from langchain_text_splitters import NLTKTextSplitter

In [ ]:
import nltk, os

download_dir = os.path.join(os.getcwd(), 'nltk_data')
nltk.download('punkt', download_dir=download_dir)
nltk.download('punkt_tab', download_dir=download_dir)
nltk.data.path.append(download_dir)

In [ ]:
text_splitter = NLTKTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)

In [ ]:
chunk_docs = text_splitter.split_documents(raw_docs)

In [ ]:
print(f"청크 개수: {len(chunk_docs)}")

for i, doc in enumerate(chunk_docs[:2]):
    print(f"\n--- 조각 {i+1} ({len(doc.page_content)}) ---\n{doc.page_content}")

## MarkdownTextSplitter
- 마크다운 텍스트를 구조적 요소(헤더(#), 리스트(-, *), 코드 블록(```))로, chunk_size 길이에 맞추어 분할합니다.

In [ ]:
from langchain_text_splitters import MarkdownTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [ ]:
# 마크다운 분할 실습용 예제 문서 (보안 권고문 형식)
sample_markdown = """# 보안 권고문: 예제 제품 취약점

## 1. 개요
예제 제품 v2.0 이하 버전에서 권한 상승 취약점이 발견되었습니다.
공격자는 이 취약점을 이용해 시스템 권한을 획득할 수 있습니다.

## 2. 영향받는 제품
- 예제 제품 v1.0 ~ v2.0
- 예제 제품 라이트 v1.5 이하

## 3. 대응 방안

### 3.1 패치 적용
공식 사이트에서 v2.1 이상으로 업데이트하십시오.

### 3.2 임시 완화 조치
- 신뢰할 수 없는 파일을 열지 마십시오.
- 안티바이러스 소프트웨어를 설치하십시오.

## 4. 문의처
보안팀에 문의하시기 바랍니다.
"""

### Markdown-Based Splitting
크기를 우선으로 텍스트를 분할 -> 마크다운 문법을 존중하며 지정된 chunk_size를 넘지 않도록 분할

In [ ]:
text_splitter = MarkdownTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)

In [ ]:
chunk_texts = text_splitter.split_text(sample_markdown)

In [ ]:
print(f"청크 개수: {len(chunk_texts)}")

for i, chunk in enumerate(chunk_texts[:3]):
    print(f"\n--- 조각 {i+1} ({len(chunk)}) ---\n{chunk}")

### Markdown Header-Based Splitting
헤더( #, ##)에 따른 구조적 조각 생성 -> headers_to_split_on에 지정된 헤더 태그가 나타날 때마다 분할

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

In [ ]:
text_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

In [ ]:
chunk_docs = text_splitter.split_text(sample_markdown)

In [ ]:
print(f"청크 개수: {len(chunk_docs)}")

for i, doc in enumerate(chunk_docs[:3]):
    print(f"\n--- 조각 {i+1} ({len(doc.page_content)}) --- {doc.metadata}\n\n{doc.page_content}")

## HTMLHeaderTextSplitter
- HTML 문서를 \<h1>, \<h2> 등 Header 태그 기준으로 분할합니다. chunk_size, chunk_overlap 파라미터를 받지 않습니다.
- 실제 웹에서 가져온 CISA 권고문 HTML(`ICSA.html`)을 그대로 분할해 봅니다.

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

In [ ]:
headers_to_split_on = [
    ("h1", "Title"),
    ("h2", "Section"),
    ("h3", "Subsection"),
    ("h4", "Details")
]

In [ ]:
text_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

In [ ]:
# html_text는 앞의 "HTML 로딩" 단계에서 읽어 둔 원본 HTML 문자열입니다.
chunk_docs = text_splitter.split_text(html_text)

In [ ]:
print(f"청크 개수: {len(chunk_docs)}")

for i, doc in enumerate(chunk_docs[:3]):
    print(f"\n--- 조각 {i+1} ({len(doc.page_content)}) --- {doc.metadata}\n\n{doc.page_content[:300]}")

## LanguageSpecificTextSplitter
- class, def (Python의 경우) 등 선택한 프로그래밍 언어의 주요 구문(클래스, 함수 정의 등)을 기준으로 분할합니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [ ]:
# 예제 파이썬 코드를 파일로 생성합니다.
sample_python = '''import math


def area_of_circle(radius: float) -> float:
    """원의 넓이를 계산합니다."""
    return math.pi * radius ** 2


def area_of_rectangle(width: float, height: float) -> float:
    """직사각형의 넓이를 계산합니다."""
    return width * height


class Shape:
    """도형의 기본 클래스."""

    def __init__(self, name: str):
        self.name = name

    def describe(self) -> str:
        return f"{self.name} 도형입니다."


class Circle(Shape):
    """원 클래스."""

    def __init__(self, radius: float):
        super().__init__("원")
        self.radius = radius

    def area(self) -> float:
        return area_of_circle(self.radius)


class Rectangle(Shape):
    """직사각형 클래스."""

    def __init__(self, width: float, height: float):
        super().__init__("직사각형")
        self.width = width
        self.height = height

    def area(self) -> float:
        return area_of_rectangle(self.width, self.height)
'''

with open("res/sample_python.py", "w", encoding="utf-8") as f:
    f.write(sample_python)

In [ ]:
with open('res/sample_python.py', 'r', encoding='utf-8') as f:
    python_code = f.read()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, 
    chunk_size=400, 
    chunk_overlap=0
)

In [ ]:
chunk_texts = text_splitter.split_text(python_code)

In [ ]:
print(f"청크 개수: {len(chunk_texts)}")

for i, chunk in enumerate(chunk_texts[:10]):
    print(f"\n--- 조각 {i+1} ({len(chunk)}) ---\n{chunk}")

# Test Splitter 적용

이제 실제 RAG에 사용할 분할 전략을 정합니다.
보안 권고문처럼 문단 구조가 있는 일반 텍스트에는 `RecursiveCharacterTextSplitter`가 무난합니다.
`chunk_overlap`을 조금 주면 청크 경계에서 문맥이 끊기는 것을 완화할 수 있습니다.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

In [ ]:
docs = text_splitter.split_documents(raw_docs)
print(f"RAG에 사용할 청크 수: {len(docs)}")

# Embedding

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
MODEL_EMBED = "intfloat/multilingual-e5-base" # sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
embedding = HuggingFaceEmbeddings(model_name=MODEL_EMBED)

In [ ]:
query = "Relion 670 시리즈에서 발견된 취약점은 무엇인가요?"

vector = embedding.embed_query(query)
print(len(vector))
print(vector[:10])

# Vector store

In [ ]:
from langchain_chroma import Chroma

In [ ]:
db = Chroma.from_documents(docs, embedding)

In [ ]:
retriever = db.as_retriever()

In [ ]:
query = "Relion 670 시리즈에서 발견된 취약점은 무엇인가요?"

context_docs = retriever.invoke(query)
print(f"len = {len(context_docs)}")

In [ ]:
first_doc = context_docs[0]
print(f"metadata = {first_doc.metadata}")
print(first_doc.page_content)

# LCEL을 사용한 RAG 구현

이제 전체를 연결합니다: 질문 → 벡터 검색(retriever) → 문맥 삽입(prompt) → 답변 생성(chat_model)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 바탕으로 질문에 한국어로 답변해 주세요.
문맥에 없는 내용은 "문서에서 찾을 수 없습니다"라고 답하세요.

문맥: """
{context}
"""

질문: {question}
''')

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

## 문서 기반 질의응답

두 권고문의 내용을 실제로 질문해 봅니다.
- HTML 문서(CISA 권고): Hitachi Energy Relion 취약점
- PDF 문서(Mitsubishi 권고): FA 소프트웨어 취약점

In [ ]:
# 질문 1: HTML 문서(CISA 권고) 내용
output = chain.invoke("Relion 670 시리즈에서 발견된 취약점의 종류와 CVSS 점수를 알려주세요.")
print(output)

In [ ]:
# 질문 2: PDF 문서(Mitsubishi 권고) 내용
output = chain.invoke("Mitsubishi FA 소프트웨어 취약점에 대한 완화 조치(Mitigations)를 알려주세요.")
print(output)

In [ ]:
# 질문 3: 문서에 없는 내용 → 환각(hallucination) 여부 확인
output = chain.invoke("삼성전자 스마트폰의 취약점에 대해 알려주세요.")
print(output)

## 정리

| 구분 | RAG 없이 | RAG 적용 |
|---|---|---|
| 지식 범위 | 모델 학습 시점까지 | 우리가 제공한 문서 |
| 근거 추적 | 불가능 | metadata(source, page)로 가능 |
| 최신성 | 고정 | 문서만 교체하면 갱신 |

작은 0.5B 모델도 정확한 문맥을 제공받으면 문서 기반 질문에 답할 수 있다는 것이 RAG의 핵심입니다.